# Chapter 4: Practice Lab - KV Caching Mechanism

I am extending my Multi-Head Attention intuition to include a Key-Value cache. This will drastically speed up autoregressive decoding.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)
print("Ready to explore KV Caching!")

## 1. Simulating the Shapes

If my Prompt is "Time flies", I have 2 tokens. The context is `B=1, T=2`.

In [ ]:
batch_size = 1
seq_len = 2
head_dim = 64

# Mocking previous K and V shapes (simulating they have already been computed)
historical_K = torch.rand(batch_size, seq_len, head_dim)
historical_V = torch.rand(batch_size, seq_len, head_dim)
print(f"Historical K shape: {historical_K.shape}")

## 2. Generating the Next Token

When calculating generation step 3 ("fast"), I only need to pass the query, key, and value for *that single new token* (`seq_len = 1`). Then I concatenate it to the history.

In [ ]:
new_token_K = torch.rand(batch_size, 1, head_dim)
new_token_V = torch.rand(batch_size, 1, head_dim)

# The crucial KV cache step: concatenation along the sequence dimension (dim=1)
updated_K_cache = torch.cat((historical_K, new_token_K), dim=1)
updated_V_cache = torch.cat((historical_V, new_token_V), dim=1)

print(f"New K Cache shape: {updated_K_cache.shape}")
# See? Now it has 3 tokens! My query will attend to all 3 without having to re-multiply them.